# Fase 2 — Entrenar YOLOv8n sobre el dataset combinado

Pensado para **Kaggle Notebooks** (GPU P100 o T4x2, "Save & Run All" corre en sus servidores aunque
cierres la pestaña — a diferencia de Colab free, que se desconecta por inactividad). También corre
en Colab sin cambios si lo preferís.

**Antes de correr:** subí `final_dataset.zip` (el que bajó `01_assemble_dataset.ipynb`) como
Kaggle Dataset ("+ Add Input" en el panel derecho de un notebook nuevo, o New Dataset desde el
menú), y poné acá abajo la ruta donde Kaggle lo monta (normalmente
`/kaggle/input/<nombre-del-dataset>/final_dataset.zip`).

In [ ]:
ZIP_PATH = "/kaggle/input/barbell-detector-dataset/final_dataset.zip"  # <- ajustar si el nombre del dataset es otro


In [ ]:
!pip install -q ultralytics


In [ ]:
import zipfile, os

DATASET_DIR = "/kaggle/working/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATASET_DIR)

data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")
print(open(data_yaml_path).read())


## Entrenamiento

`epochs=300` + `patience=50` (early stopping): corta solo si no mejora en 50 épocas, así no
desperdicia las ~6h si converge antes, y no se corta antes de tiempo si todavía está mejorando.
YOLOv8n ("nano") a propósito — es la variante más chica, pensada para exportar a celular después
(Fase 4), no la más precisa posible.

In [ ]:
from ultralytics import YOLO
import time

model = YOLO("yolov8n.pt")

start = time.time()
results = model.train(
    data=data_yaml_path,
    epochs=300,
    patience=50,
    imgsz=640,
    batch=-1,  # ultralytics elige el batch size mas grande que entra en la GPU disponible
    device=0,
    project="/kaggle/working/runs",
    name="barbell_detector",
)
elapsed_hours = (time.time() - start) / 3600
print(f"Entrenamiento terminado en {elapsed_hours:.2f} horas")


## Fase 3 — Resultados reales sobre el set de validación (nunca sobre training)

`model.val()` corre de nuevo sobre el split `valid/` definido en `data.yaml` — son los números que
importan, no los que salen durante el entrenamiento (esos pueden estar inflados si hubo overfitting).

In [ ]:
best_model = YOLO("/kaggle/working/runs/barbell_detector/weights/best.pt")
metrics = best_model.val(data=data_yaml_path, split="val")

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print("\nPor clase (mAP50):")
for i, name in metrics.names.items():
    print(f"  {name}: {metrics.box.ap50[i]:.4f}")


## Ejemplos reales de detecciones (buenas y malas) sobre el set de test

No solo el número — mirá estas imágenes antes de dar por bueno el modelo.

In [ ]:
import glob, random
from PIL import Image

test_images = glob.glob(os.path.join(DATASET_DIR, "test", "images", "*"))
sample = random.sample(test_images, k=min(10, len(test_images)))
pred_results = best_model.predict(sample, conf=0.25, save=True, project="/kaggle/working/runs", name="test_preview")

for r in pred_results:
    print(os.path.basename(r.path), "->", len(r.boxes), "detecciones")
    display(Image.fromarray(r.plot()[:, :, ::-1]).resize((480, 360)))


## Fase 4 (solo investigación) — exportar a ONNX, para medir si es viable en navegador

No se integra a la app todavía — esto solo confirma el peso real del archivo. Si pesa varios
cientos de MB, hay que reconsiderar antes de cargarlo en una página web.

In [ ]:
onnx_path = best_model.export(format="onnx", imgsz=640)
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX exportado: {onnx_path} — {size_mb:.1f} MB")


## Empaquetar todo para descargar

Con "Save & Run All" (commit), esto queda en el tab **Output** del notebook — no hace falta tener
la pestaña abierta mientras entrena, solo volver después y descargar `resultado_entrenamiento.zip`.

In [ ]:
import shutil

OUT_DIR = "/kaggle/working/resultado_entrenamiento"
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy("/kaggle/working/runs/barbell_detector/weights/best.pt", OUT_DIR)
shutil.copy(onnx_path, OUT_DIR)
for plot in glob.glob("/kaggle/working/runs/barbell_detector/*.png"):
    shutil.copy(plot, OUT_DIR)
shutil.copytree("/kaggle/working/runs/test_preview", os.path.join(OUT_DIR, "test_preview"), dirs_exist_ok=True)

with open(os.path.join(OUT_DIR, "resumen.txt"), "w") as f:
    f.write(f"mAP50: {metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
    f.write(f"Horas de entrenamiento: {elapsed_hours:.2f}\n")
    f.write(f"Tamaño ONNX: {size_mb:.1f} MB\n")

shutil.make_archive("/kaggle/working/resultado_entrenamiento", "zip", OUT_DIR)
print("Listo: /kaggle/working/resultado_entrenamiento.zip")
